In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import seaborn as sns
from tqdm import tqdm
load_dotenv()

True

In [2]:
responses_path=os.getenv("RESPONSES_PROCESSED_PATH")
df=pd.read_csv(responses_path)
df = df.sort_values(['uid','timestamps'])  

In [3]:
responses_path=os.getenv("RESPONSES_TRAIN_PATH")
responses_path=os.path.join(responses_path,"train_valid_sequences.csv")

training_set=pd.read_csv(responses_path)

df = training_set.copy()

for col in ['questions', 'concepts', 'responses','timestamps']:
    if isinstance(df[col].iloc[0], str):
        # If stored as comma-separated string
        df[col] = df[col].str.split(',')
    # If it's already a list, keep it

# explode all columns simultaneously
df = df.explode(['questions', 'concepts', 'responses','timestamps'])

# Convert columns to appropriate types
df['questions'] = df['questions'].astype(int)
df['concepts'] = df['concepts'].astype(int)
df['responses'] = df['responses'].astype(int)

# Filter out padding responses (-1)
df = df[df['responses'] != -1]
df=df[df['questions'] != -1]
df=df[df['concepts'] != -1]

df=df.drop(['selectmasks','is_repeat'],axis=1)


In [ ]:
# ============================================================
# KC Learning Curves (Error Rate by Exposure Number)
# ============================================================

# For each user-KC pair, count how many times they've seen this KC
df['kc_exposure'] = df.groupby(['uid', 'concepts']).cumcount() + 1

# For each KC, calculate error rate at each exposure level
kc_learning = df.groupby(['concepts', 'kc_exposure']).agg(
    attempted=('responses', 'count'),
    correct=('responses', lambda x: (x == 1).sum()),
    wrong=('responses', lambda x: (x == 0).sum())
).reset_index()

kc_learning['error_rate'] = kc_learning['wrong'] / kc_learning['attempted']

# Pivot to get KC as rows, exposure number as columns
kc_learning_pivot = kc_learning.pivot(
    index='concepts',
    columns='kc_exposure',
    values='error_rate'
).fillna(np.nan)

# Add metadata: which KC each row represents
kc_learning_pivot.index.name = 'kc_id'
kc_learning_pivot = kc_learning_pivot.reset_index()

# Rename columns to indicate exposure number
kc_learning_pivot.columns = ['kc_id'] + [f'exposure_{i}' for i in range(1, len(kc_learning_pivot.columns))]

print("KC Learning Curves (Error Rate by Exposure Number):")
print(kc_learning_pivot.head())
print(f"Shape: {kc_learning_pivot.shape}")
kc_learning_pivot.to_csv('practice_effect_perKC.csv',index=False)

KC Learning Curves (Error Rate by Exposure Number):
   kc_id  exposure_1  exposure_2  exposure_3  exposure_4  exposure_5  \
0      0    0.161626    0.478320    0.514493    0.142857    0.333333   
1      1    0.237633    0.202247    0.108374    0.068966    0.000000   
2      2    0.092586    0.105867    0.076322    0.083297    0.050448   
3      3    0.220941    0.215200    0.354418    0.338787    0.399425   
4      4    0.105606         NaN         NaN         NaN         NaN   

   exposure_6  exposure_7  exposure_8  exposure_9  ...  exposure_47  \
0         NaN         NaN         NaN         NaN  ...          NaN   
1    0.000000         NaN         NaN         NaN  ...          NaN   
2    0.042087    0.037657    0.100000         0.4  ...          NaN   
3    0.252427    0.423729    0.285714         0.8  ...          NaN   
4         NaN         NaN         NaN         NaN  ...          NaN   

   exposure_48  exposure_49  exposure_50  exposure_51  exposure_52  \
0          NaN    

In [ ]:
# ============================================================
#  Question Learning Curves 
# ============================================================

question_kc_map = df.groupby('questions')['concepts'].apply(set).reset_index()
question_kc_map.columns = ['question_id', 'kc_set']

# Add sequence position (which question in the user's sequence)
df['seq_pos'] = df.groupby('uid').cumcount() + 1

# For each question, find the error rate of its KCs at each sequence position
def practice_effect_PerQ(df, question_kc_map):
    """Vectorized approach - much faster"""
    
    # Create a set of all unique sequence positions
    all_seq = sorted(df['seq_pos'].unique())
    
    results = []
    
    # Pre-compute cumulative error rates for each KC
    # This is the key optimization!
    kc_cumulative = {}
    for kc in df['concepts'].unique():
        kc_data = df[df['concepts'] == kc].sort_values(['uid', 'seq_pos'])
        kc_data['cum_error_rate'] = kc_data.groupby('uid')['responses'].transform(
            lambda x: (x == 0).expanding().mean()
        )
        kc_cumulative[kc] = kc_data[['uid', 'seq_pos', 'cum_error_rate']]
    
    # For each question, average the cumulative error rates of its KCs
    for _, row in question_kc_map.iterrows():
        q_id = row['question_id']
        kc_set = row['kc_set']
        
        # Get cumulative error rates for all KCs in this question
        kc_dfs = [kc_cumulative[kc] for kc in kc_set if kc in kc_cumulative]
        if not kc_dfs:
            continue
            
        # Merge all KC data by uid and seq_pos
        combined = kc_dfs[0]
        for kc_df in kc_dfs[1:]:
            combined = combined.merge(
                kc_df,
                on=['uid', 'seq_pos'],
                suffixes=('', '_temp')
            )
        
        # Average across KCs
        error_cols = [col for col in combined.columns if col.startswith('cum_error_rate')]
        combined['avg_error'] = combined[error_cols].mean(axis=1)
        
        # Aggregate by sequence position
        seq_avg = combined.groupby('seq_pos')['avg_error'].mean().reset_index()
        seq_avg['question_id'] = q_id
        
        results.append(seq_avg)
    
    # Combine all results
    final_df = pd.concat(results, ignore_index=True)
    
    # Pivot
    pivot = final_df.pivot(
        index='question_id',
        columns='seq_pos',
        values='avg_error'
    ).fillna(np.nan)
    
    pivot = pivot.reset_index()
    pivot.columns = ['question_id'] + [f'after_{i}_qs' for i in range(1, len(pivot.columns))]
    
    return pivot
practice_effect = get_practice_effect_PerQ(df, question_kc_map)
question_learning_pivot.to_csv('practice_effect_perQ.csv',index=False)

In [ ]:
def diffrentiation_static(df):

    print("Pre-computing KC error rates per student...")
    
    # For each student-KC pair, compute their overall error rate
    student_kc_error = df.groupby(['uid', 'concepts'])['responses'].agg(
        error_rate=lambda x: (x == 0).mean(),
        count='count'
    ).reset_index()
    
    print("Merging question data...")

    # For each question, get its KCs and merge with student-KC error rates
    q_with_kcs = df[['uid', 'questions', 'concepts', 'responses', 'seq_pos']].copy()
    
    # Merge to get each student's baseline error rate for each KC
    q_with_kcs = q_with_kcs.merge(
        student_kc_error,
        on=['uid', 'concepts'],
        how='left'
    )
    
    # For each question, calculate the delta between the student's response
    # and their baseline error rate on that KC
    q_with_kcs['delta'] = (1 - q_with_kcs['responses']) - q_with_kcs['error_rate']
    
    print("Aggregating per question...")
    
    # Aggregate per question
    result = q_with_kcs.groupby('questions').agg(
        avg_delta=('delta', 'mean'),
        n_students=('uid', 'count'),
        improvement_rate=('delta', lambda x: (x < 0).mean()),
        q_error_rate=('responses', lambda x: (x == 0).mean()),
        avg_baseline_error=('error_rate', 'mean')
    ).reset_index()
    
    return result

question_diffrentiation = diffrentiation_static(df)

In [ ]:
question_impact.to_csv('question_diffrentiation.csv',index=False)

In [ ]:

def cumulative_diffrentiation(df):

    # Sort by user and timestamp
    df = df.sort_values(['uid']).reset_index(drop=True)
    
    # Convert response to error indicator (1=wrong, 0=correct)
    df['error'] = 1 - df['responses']
    
    # ============================================================
    # STEP 1: Compute cumulative error rate for each KC per student
    # ============================================================
    print("Step 1/4: Computing cumulative KC error rates per student...")
    
    # This is the magic - for each user-KC pair, compute expanding mean
    # The shift() ensures we use ONLY previous responses (not current)
    df['kc_error_cumulative'] = df.groupby(['uid', 'concepts'])['error'].transform(
        lambda x: x.expanding().mean().shift()
    )
    
    # For first exposure to a KC, there is no baseline - set to NaN
    # We'll fill with the student's overall error rate later
    df['kc_error_cumulative'] = df.groupby(['uid', 'concepts'])['kc_error_cumulative'].transform(
        lambda x: x.fillna(x.mean())
    )
    
    # ============================================================
    # STEP 2: Calculate delta for each response
    # ============================================================
    print("Step 2/4: Computing deltas per response...")
    
    # Delta = current error - cumulative error before this question
    # Negative = improvement over baseline
    df['delta'] = df['error'] - df['kc_error_cumulative']
    
    # ============================================================
    # STEP 3: For each question, get its KCs (explode)
    # ============================================================
    print("Step 3/4: Building question-KC mapping...")
    
    # Get unique question-KC pairs (avoid set() for speed)
    q_kc_pairs = df[['questions', 'concepts']].drop_duplicates()
    
    # ============================================================
    # STEP 4: Merge and aggregate per question
    # ============================================================
    print("Step 4/4: Computing per-question learning metrics...")
    
    # Merge delta info with question-KC mapping
    merged = df[['uid', 'questions', 'concepts', 'delta', 'error', 'kc_error_cumulative']].merge(
        q_kc_pairs,
        on=['questions', 'concepts'],
        how='inner'
    )
    
    # Aggregate per question
    question_impact = merged.groupby('questions').agg(
        avg_delta=('delta', 'mean'),
        median_delta=('delta', 'median'),
        improvement_rate=('delta', lambda x: (x < 0).mean()),
        worsening_rate=('delta', lambda x: (x > 0).mean()),
        n_observations=('delta', 'count'),
        n_students=('uid', 'nunique'),
        avg_baseline=('kc_error_cumulative', 'mean'),
        question_error=('error', 'mean')
    ).reset_index()
    
    # diffrentiation effect (positive = learning)
    question_impact['diffrentiation_effect'] = -question_impact['avg_delta']
    question_impact['relative_improvement'] = question_impact['diffrentiation_effect'] / (question_impact['avg_baseline'] + 0.001)
    
    return question_impact, df

question_impact, df_with_errors = cumulative_diffrentiation(df)
question_impact.to_csv('question_diffrentiation_cumulative.csv', index=False)


Starting ultra-fast calculation...


NameError: name 'df' is not defined

In [4]:
# This is the fast way - should run in ~2 minutes
df_sorted = df.sort_values(['uid', 'timestamps'])
df_sorted['error'] = 1 - df_sorted['responses']
df_sorted['prev_error'] = df_sorted.groupby(['uid', 'concepts'])['error'].transform(
    lambda x: x.expanding().mean().shift()
)

In [ ]:
df_sorted['next_error'] = df_sorted.groupby(['uid', 'concepts'])['error'].shift(-1)
# Aggregator for the learning delta
chronological_delta = df_sorted.groupby('questions').apply(
    lambda group: pd.Series({
        'before_error': group['prev_error'].mean(),
        'after_error': group['next_error'].mean(),
        'learning_delta': group['next_error'].mean() - group['prev_error'].mean(),
        'improvement_rate': (group['next_error'] < group['prev_error']).mean()
    })
).reset_index()

print(chronological_delta.head())

   questions  before_error  after_error  learning_delta  improvement_rate
0          0      0.583333     0.524213       -0.059120          0.000000
1          1      0.116141     0.040073       -0.076068          0.055027
2          2      0.216514     0.366573        0.150059          0.073272
3          3      0.227585     0.427743        0.200158          0.036189
4          4           NaN          NaN             NaN          0.000000


C:\Users\Hatem\AppData\Local\Temp\ipykernel_20980\2557894597.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  chronological_delta = df_sorted.groupby('questions').apply(


In [8]:
chronological_delta.to_csv('chronological_delta.csv',index=False)

In [9]:
print(chronological_delta.head(15))

    questions  before_error  after_error  learning_delta  improvement_rate
0           0      0.583333     0.524213       -0.059120          0.000000
1           1      0.116141     0.040073       -0.076068          0.055027
2           2      0.216514     0.366573        0.150059          0.073272
3           3      0.227585     0.427743        0.200158          0.036189
4           4           NaN          NaN             NaN          0.000000
5           5      0.170374     0.212766        0.042392          0.044415
6           6      0.210006     0.220093        0.010087          0.038685
7           7      0.236622     0.047306       -0.189316          0.013627
8           8      0.177333     0.187692        0.010359          0.033186
9           9      0.213396     0.177358       -0.036037          0.090323
10         10      0.225667     0.174377       -0.051289          0.082883
11         11      0.156250     0.145695       -0.010555          0.000000
12         12      0.1289